# Week6_ex3 - 4-Pole 3-Slot IPM Machine, Rotor Angle Sweep

Homework exercise on a 4-pole/3-slot interior permanent magnet machine -- rotates the rotor in 10-degree steps over one full turn, feeding each coil a 120-degree-phase-shifted current, and logs torque/force at every step by physically rotating the stator geometry and re-analyzing.

**Known limitation:** this design does not solve on the Student license. The full 360-degree model consistently exceeds Maxwell 2D's 2,000-element mesh cap regardless of mesh coarsening (root cause: the stator's slot profile geometry itself, not the adaptive refinement settings). A symmetry-reduced model isn't a viable workaround either -- 4 poles and 3 slots share no common divisor, so no rotational symmetry fraction (1/2, 1/4, ...) applies to the geometry, and even a slot count that *would* support symmetry (e.g. 6 or 12 slots) still fails: the three-phase winding is not mirror-symmetric (phases A/B/C are distinct time-varying waveforms, not spatial reflections of each other), so a simple mirror symmetry boundary produces physically invalid results, and a correct fix would need rotationally-periodic (Master/Slave) boundaries plus a from-scratch winding layout -- out of scope here.


In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M2D = ansys.aedt.core.maxwell.Maxwell2d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M2D.odesign
# oproject object for script-recording style calls
oProject = M2D.oproject

In [ ]:
## Set up output directory for results ##

proj_name = "Week6_ex3"   # matches this exercise's folder name

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week6_ex3"

In [ ]:
## Save project and apply design name ##

proj = M2D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M2D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M2D.save_project()

In [ ]:
## Design variable definitions ##

# base dimensions
depth = 100         # stack length of the model
M2D["depth"] = f"{depth}mm"

slot_number = 3     # number of slots
M2D["slot_number"] = f"{slot_number}"   # count, no unit

r_so = 100          # stator outer radius
M2D["r_so"] = f"{r_so}mm"

r_si = 60           # stator inner radius -- must satisfy r_so - r_rotor_outer > r_si
M2D["r_si"] = f"{r_si}mm"

th_core = 2         # stator core thickness
M2D["th_core"] = f"{th_core}mm"

w_teeth = 20        # distance between teeth
M2D["w_teeth"] = f"{w_teeth}mm"

slot_ratio = 0.25   # slot opening ratio
M2D["slot_ratio"] = f"{slot_ratio}"   # ratio, no unit

shoe_1 = 10         # slot shoe size 1
M2D["shoe_1"] = f"{shoe_1}mm"

shoe_2 = 10         # slot shoe size 2
M2D["shoe_2"] = f"{shoe_2}mm"


# coil dimensions
c_w = 10            # coil width
M2D["c_w"] = f"{c_w}mm"

c_h = 10            # coil height
M2D["c_h"] = f"{c_h}mm"

l_c = 85            # coil position, measured from origin (0,0)
M2D["l_c"] = f"{l_c}mm"


# rotor dimensions
r_rotor_outer = 55  # rotor outer radius
M2D["r_rotor_outer"] = f"{r_rotor_outer}mm"


# magnet dimensions
num_pole = 4        # number of rotor magnet poles
M2D["num_pole"] = f"{num_pole}"   # count, no unit

m_w = 30            # magnet width
M2D["m_w"] = f"{m_w}mm"

m_th = 10           # magnet thickness
M2D["m_th"] = f"{m_th}mm"

l_m = 40            # magnet position, measured from origin (0,0)
M2D["l_m"] = f"{l_m}mm"


# rotating field / current density variables
gamma = 0           # current angle -- fixed at 0 deg for SPM-style control
M2D["gamma"] = f"{gamma}deg"

N = 36              # number of steps to sweep from 0 to 360 degrees
M2D["N"] = f"{N}"

J_rated = 3         # coil region current density
M2D["J_rated"] = f"{J_rated}"   # add units, e.g. f"{J_rated}A/mm^2", if needed later


# three-phase current -- set from inside the sweep loop below
current_a = 0
M2D["current_a"] = f"{current_a}A"

current_b = 0
M2D["current_b"] = f"{current_b}A"

current_c = 0
M2D["current_c"] = f"{current_c}A"

In [ ]:
## Material definitions ##

# 1. magnets -- one duplicate per pole, magnetized toward the direction that pole points
magnet_1 = M2D.materials.duplicate_material("NdFe35", name="magnet_1")
magnet_1.get_magnetic_coercivity()  # check magnetization
magnet_1.set_magnetic_coercivity(value=-900000, x=0, y=1, z=0) 

magnet_2 = M2D.materials.duplicate_material("NdFe35", name="magnet_2")
magnet_2.get_magnetic_coercivity()  # check magnetization
magnet_2.set_magnetic_coercivity(value=-900000, x=1, y=0, z=0)

magnet_3 = M2D.materials.duplicate_material("NdFe35", name="magnet_3")
magnet_3.get_magnetic_coercivity()  # check magnetization
magnet_3.set_magnetic_coercivity(value=-900000, x=0, y=-1, z=0)

magnet_4 = M2D.materials.duplicate_material("NdFe35", name="magnet_4")
magnet_4.get_magnetic_coercivity()  # check magnetization
magnet_4.set_magnetic_coercivity(value=-900000, x=-1, y=0, z=0)



# 2. coil copper
copper_c = M2D.materials.duplicate_material("copper", name="copper_c")   # coil copper



# 3. stator/rotor steel (includes BH curve data)
steel = M2D.materials.duplicate_material("steel_1008", name="steel")

# - using the raw script-recording API here since duplicate_material() alone can't set a nonlinear BH curve
oDefinitionManager = oProject.GetDefinitionManager()
oDefinitionManager.EditMaterial("steel", 
	[
		"NAME:steel",
		"CoordinateSystemType:=", "Cartesian",
		"BulkOrSurfaceType:="	, 1,
		[
			"NAME:PhysicsTypes",
			"set:="			, ["Electromagnetic","Thermal","Structural"]
		],
		[
			"NAME:AttachedData",
			[
				"NAME:MatAppearanceData",
				"property_data:="	, "appearance_data",
				"Red:="			, 161,
				"Green:="		, 161,
				"Blue:="		, 161
			]
		],
		"permittivity:="	, "1",
		[
			"NAME:permeability",
			"property_type:="	, "nonlinear",
			"BTypeForSingleCurve:="	, "normal",
			"HUnit:="		, "A_per_meter",
			"BUnit:="		, "tesla",
			"IsTemperatureDependent:=", False,
			[
				"NAME:BHCoordinates",
				[
					"NAME:DimUnits", 
					"", 
					""
				],
				[
					"NAME:Point", 
					0, 
					0
				],
				[
					"NAME:Point", 
					15.120714, 
					0.05
				],
				[
					"NAME:Point", 
					22.718292, 
					0.1
				],
				[
					"NAME:Point", 
					27.842733, 
					0.15
				],
				[
					"NAME:Point", 
					31.871434, 
					0.2
				],
				[
					"NAME:Point", 
					35.365044, 
					0.25
				],
				[
					"NAME:Point", 
					38.600588, 
					0.3
				],
				[
					"NAME:Point", 
					41.736202, 
					0.35
				],
				[
					"NAME:Point", 
					44.873979, 
					0.4
				],
				[
					"NAME:Point", 
					48.087807, 
					0.45
				],
				[
					"NAME:Point", 
					51.437236, 
					0.5
				],
				[
					"NAME:Point", 
					54.975221, 
					0.55
				],
				[
					"NAME:Point", 
					58.752993, 
					0.6
				],
				[
					"NAME:Point", 
					62.823644, 
					0.65
				],
				[
					"NAME:Point", 
					67.245285, 
					0.7
				],
				[
					"NAME:Point", 
					72.084406, 
					0.75
				],
				[
					"NAME:Point", 
					77.4201, 
					0.8
				],
				[
					"NAME:Point", 
					83.350021, 
					0.85
				],
				[
					"NAME:Point", 
					89.999612, 
					0.9
				],
				[
					"NAME:Point", 
					97.537353, 
					0.95
				],
				[
					"NAME:Point", 
					106.201406, 
					1
				],
				[
					"NAME:Point", 
					116.348464, 
					1.05
				],
				[
					"NAME:Point", 
					128.547329, 
					1.1
				],
				[
					"NAME:Point", 
					143.765431, 
					1.15
				],
				[
					"NAME:Point", 
					163.754169, 
					1.2
				],
				[
					"NAME:Point", 
					191.868158, 
					1.25
				],
				[
					"NAME:Point", 
					234.833507, 
					1.3
				],
				[
					"NAME:Point", 
					306.509769, 
					1.35
				],
				[
					"NAME:Point", 
					435.255202, 
					1.4
				],
				[
					"NAME:Point", 
					674.911968, 
					1.45
				],
				[
					"NAME:Point", 
					1108.325569, 
					1.5
				],
				[
					"NAME:Point", 
					1813.085468, 
					1.55
				],
				[
					"NAME:Point", 
					2801.217421, 
					1.6
				],
				[
					"NAME:Point", 
					4053.653117, 
					1.65
				],
				[
					"NAME:Point", 
					5591.10689, 
					1.7
				],
				[
					"NAME:Point", 
					7448.318413, 
					1.75
				],
				[
					"NAME:Point", 
					9708.81567, 
					1.8
				],
				[
					"NAME:Point", 
					12486.931615, 
					1.85
				],
				[
					"NAME:Point", 
					16041.483644, 
					1.9
				],
				[
					"NAME:Point", 
					21249.420624, 
					1.95
				],
				[
					"NAME:Point", 
					31313.495878, 
					2
				],
				[
					"NAME:Point", 
					53589.446877, 
					2.05
				],
				[
					"NAME:Point", 
					88477.484601, 
					2.1
				],
				[
					"NAME:Point", 
					124329.41054, 
					2.15
				],
				[
					"NAME:Point", 
					159968.5693, 
					2.2
				],
				[
					"NAME:Point", 
					197751.604272, 
					2.25
				],
				[
					"NAME:Point", 
					234024.751347, 
					2.3
				],
			],
			[
				"NAME:Temperatures"
			],
			"RecoilPermeability:="	, "1"
		],
		"conductivity:="	, "2000000",
		"dielectric_loss_tangent:=", "0",
		"magnetic_loss_tangent:=", "0",
		[
			"NAME:magnetic_coercivity",
			"property_type:="	, "VectorProperty",
			"Magnitude:="		, "0A_per_meter",
			"DirComp1:="		, "1",
			"DirComp2:="		, "0",
			"DirComp3:="		, "0"
		],
		"thermal_conductivity:=", "45",
		"mass_density:="	, "7872",
		[
			"NAME:stacking_type",
			"property_type:="	, "ChoiceProperty",
			"Choice:="		, "Lamination"
		],
		"stacking_factor:="	, "0.98",
		[
			"NAME:stacking_direction",
			"property_type:="	, "ChoiceProperty",
			"Choice:="		, "V(3)"
		],
		"specific_heat:="	, "481",
		"youngs_modulus:="	, "200000000000",
		"poissons_ratio:="	, "0.25",
		"thermal_expansion_coefficient:=", "1.26e-05",
		"diffusivity:="		, "0.8",
		"molecular_mass:="	, "0",
		"viscosity:="		, "0"
	])



In [ ]:
## Analyze Setup ##

# NOTE: MaximumPasses reduced 20 -> 10 and PercentRefinement dropped 30 -> 15 --
# at the original settings, every angle step in the 36-step sweep below hit the
# Student license's 2D mesh cap (~3,600-3,700 surface elements) and failed outright
# with an execution error, rather than completing with just a convergence warning
# like the t_sample sweeps elsewhere in this project. There's no manual mesh
# sizing on any object here, so fewer/coarser adaptive passes is the only lever.

# model depth (z-direction thickness for a 2D model)
M2D.model_depth = "depth"

# Magnetostatic setup, including the nonlinear BH curve option (MuOption) --
# same solver settings as the original script-recorded setup, just expressed
# through create_setup()+props instead of the raw InsertSetup call
my_setup = M2D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 10
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 15
my_setup.props["SolveFieldOnly"] = False
my_setup.props["PercentError"] = 1
my_setup.props["SolveMatrixAtLast"] = True
my_setup.props["UseNonLinearIterNum"] = False
my_setup.props["NonLinearResidual"] = 0.0001
my_setup.props["RelaxationFactor"] = 1
my_setup.props["SmoothBHCurve"] = False
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()

In [ ]:
## Model units ##

M2D.modeler.model_units = "mm"

In [ ]:
## Draw the rotor ##

# 1. magnets
origin = ["-m_w/2", "l_m-m_th/2"]
sizes = ["m_w", "m_th"]
magnet_1 = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="magnet_1", material=magnet_1, non_model=False)
magnet_1.material_name = "magnet_1"
magnet_1.color = (255, 0, 0)

# - rotate-duplicate
M2D.modeler.duplicate_around_axis(assignment=magnet_1, axis="Z", angle=90, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
# - grab the most recently created object (the duplicated magnet)
magnet_2 = M2D.modeler.object_list[-1]
magnet_2.name = "magnet_2"
magnet_2.material_name = "magnet_2"
magnet_2.color = (0, 0, 255)

M2D.modeler.duplicate_around_axis(assignment=magnet_1, axis="Z", angle=180, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
magnet_3 = M2D.modeler.object_list[-1]
magnet_3.name = "magnet_3"
magnet_3.material_name = "magnet_3"
magnet_3.color = (255, 0, 0)

M2D.modeler.duplicate_around_axis(assignment=magnet_1, axis="Z", angle=270, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
magnet_4 = M2D.modeler.object_list[-1]
magnet_4.name = "magnet_4"
magnet_4.material_name = "magnet_4"
magnet_4.color = (0, 0, 255)



# 2. rotor body
origin = [0, 0, 0]
radius = "r_rotor_outer"
# rotor = M2D.modeler.create_circle(origin=origin, radius=radius, num_sides=360, is_covered=True, name="rotor", material="vacuum", non_model=False,)
rotor = M2D.modeler.create_circle(origin=origin, radius=radius, num_sides=360, is_covered=True, name="rotor", material="steel", non_model=False,)
M2D.modeler.subtract(blank_list=rotor, tool_list=[magnet_1, magnet_2, magnet_3, magnet_4], keep_originals=True)

In [ ]:
## Draw the stator and coils ##

# 1. stator inner-line surface

# - draw the 4-point teeth line
points = [
    [-(r_so-th_core)*math.cos((math.pi/2-(math.asin(w_teeth*0.5/(r_so-th_core))))), (r_so-th_core)*math.sin((math.pi/2-(math.asin(w_teeth*0.5/(r_so-th_core))))), 0],
    [-(r_si+shoe_2+shoe_1)*math.cos((math.pi/2-(math.asin(w_teeth*0.5/(r_si+shoe_2+shoe_1))))), (r_si+shoe_2+shoe_1)*math.sin((math.pi/2-(math.asin(w_teeth*0.5/(r_si+shoe_2+shoe_1))))), 0],
    [(r_si+shoe_2)*math.cos((math.pi/2+math.pi/slot_number*slot_ratio)), (r_si+shoe_2)*math.sin((math.pi/2+math.pi/slot_number*slot_ratio)), 0],
    [r_si*math.cos((math.pi/2+math.pi/slot_number*slot_ratio)), r_si*math.sin((math.pi/2+math.pi/slot_number*slot_ratio)), 0],
]
stator_line = M2D.modeler.create_polyline(points=points, segment_type=None, cover_surface=False, close_surface=False, name="stator_line", material="vacuum", non_model=False)

# - draw the 3-point arc line inside the slot
points = [
    [-(r_so-th_core)*math.cos((math.pi/2-(math.asin(w_teeth*0.5/(r_so-th_core))))), (r_so-th_core)*math.sin((math.pi/2-(math.asin(w_teeth*0.5/(r_so-th_core))))), 0],
    [-(r_so-th_core)*math.cos(math.pi/2-math.pi/6), (r_so-th_core)*math.sin(math.pi/2-math.pi/6), 0],
    [-(r_so-th_core)*math.cos(math.pi/2-math.pi/3), (r_so-th_core)*math.sin(math.pi/2-math.pi/3), 0],
]
stator_tmp = M2D.modeler.create_polyline(points=points, segment_type="Arc", cover_surface=False, close_surface=False, name="stator_tmp", material="vacuum", non_model=False)

# - unite the line and the arc line
M2D.modeler.unite(assignment=[stator_line, stator_tmp], purge=False, keep_originals=False)

# - mirror-duplicate across the y axis, then unite
origin = [0, 0, 0]
vector = [1, 0, 0]
M2D.modeler.duplicate_and_mirror(assignment=stator_line, origin=origin, vector=vector, is_3d_comp=False, duplicate_assignment=True)
stator_tmp = M2D.modeler.object_list[-1]

M2D.modeler.unite(assignment=[stator_line, stator_tmp], purge=False, keep_originals=False)


# - draw the 3-point arc line on the airgap side, then unite
points = [
    [r_si*math.cos((math.pi/2+math.pi/slot_number*slot_ratio)), r_si*math.sin((math.pi/2+math.pi/slot_number*slot_ratio)), 0],
    [0, r_si, 0],
    [r_si*math.cos((math.pi/2-math.pi/slot_number*slot_ratio)), r_si*math.sin((math.pi/2+math.pi/slot_number*slot_ratio)), 0],
]
stator_tmp = M2D.modeler.create_polyline(points=points, segment_type="Arc", cover_surface=False, close_surface=False, name="stator_tmp", material="vacuum", non_model=False)

M2D.modeler.unite(assignment=[stator_line, stator_tmp], purge=False, keep_originals=False)


# - rotate-duplicate, grab the newest objects, then unite
M2D.modeler.duplicate_around_axis(assignment=stator_line, axis="Z", angle=120, clones=3, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)

stator_tmp1 =  M2D.modeler.object_list[-2]
stator_tmp2 =  M2D.modeler.object_list[-1]

M2D.modeler.unite(assignment=[stator_line, stator_tmp1, stator_tmp2], purge=False, keep_originals=False)


# - cover the stator inner line into a surface
M2D.modeler.cover_lines(assignment=stator_line)



# 2. draw the stator outer-line surface and subtract
origin = [0, 0, 0]
radius = "r_so"
# stator = M2D.modeler.create_circle(origin=origin, radius=radius, num_sides=360, is_covered=True, name="stator", material="vacuum", non_model=False,)
stator = M2D.modeler.create_circle(origin=origin, radius=radius, num_sides=360, is_covered=True, name="stator", material="steel", non_model=False,)


M2D.modeler.subtract(blank_list=stator, tool_list=[stator_line], keep_originals=False)



# 3. draw the coil, positive direction

origin = ["-w_teeth/2-c_w", "l_c-c_h/2"]
sizes = ["c_w", "c_h"]
coil_a_posi = M2D.modeler.create_rectangle(origin=origin, sizes=sizes, is_covered=True, name="coil_a_posi", material=copper_c, non_model=False)

# - rotate-duplicate
M2D.modeler.duplicate_around_axis(assignment=coil_a_posi, axis="Z", angle=120, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
coil_c_posi = M2D.modeler.object_list[-1]
coil_c_posi.name = "coil_c_posi"

M2D.modeler.duplicate_around_axis(assignment=coil_a_posi, axis="Z", angle=240, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
coil_b_posi = M2D.modeler.object_list[-1]
coil_b_posi.name = "coil_b_posi"



# 4. draw the coil, negative direction (mirror of positive)
origin = [0, 0, 0]
vector = [1, 0, 0]
M2D.modeler.duplicate_and_mirror(assignment=coil_a_posi, origin=origin, vector=vector, is_3d_comp=False, duplicate_assignment=True)
coil_a_nega = M2D.modeler.object_list[-1]
coil_a_nega.name = "coil_a_nega"

M2D.modeler.duplicate_around_axis(assignment=coil_a_nega, axis="Z", angle=120, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
coil_c_nega = M2D.modeler.object_list[-1]
coil_c_nega.name = "coil_c_nega"

M2D.modeler.duplicate_around_axis(assignment=coil_a_nega, axis="Z", angle=240, clones=2, create_new_objects=True, is_3d_comp=False, duplicate_assignment=True)
coil_b_nega = M2D.modeler.object_list[-1]
coil_b_nega.name = "coil_b_nega"

In [ ]:
## Region and boundary ##

# 1. create the region
region = M2D.modeler.create_region(pad_value=200, pad_type='Percentage Offset', name='Region')



# 2. assign balloon boundary
M2D.assign_balloon(assignment=region.edges, boundary=None)

In [ ]:
## Mesh setup ##

# NOTE: pyaedt's assign_length_mesh() convenience wrapper ran without error but had
# zero measurable effect -- element counts came out identical to the unconstrained
# case on every angle step. Same pattern as the other pyaedt-wrapper-doesn't-work
# issues elsewhere in this project (create_setup(), analyze(), assign_radiation()) --
# switching to the raw AssignLengthOp API instead, applied to the named solid objects
# directly (not the region/air background, which doesn't need constraining) with a
# more generous 10mm cap.
oModule = oDesign.GetModule("MeshSetup")
oModule.AssignLengthOp(
	[
		"NAME:global_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["stator", "rotor", "magnet_1", "magnet_2", "magnet_3", "magnet_4",
						   "coil_a_posi", "coil_a_nega", "coil_b_posi", "coil_b_nega", "coil_c_posi", "coil_c_nega"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "10mm"
	])

In [ ]:
## Excitation ##

# 1. assign coil terminals to the relevant objects
coil_a_posi_ter = M2D.assign_coil(assignment=coil_a_posi, conductors_number=1, polarity='Positive', name="coil_a_posi_ter")
coil_a_nega_ter = M2D.assign_coil(assignment=coil_a_nega, conductors_number=1, polarity='Negative', name="coil_a_nega_ter")
coil_b_posi_ter = M2D.assign_coil(assignment=coil_b_posi, conductors_number=1, polarity='Positive', name="coil_b_posi_ter")
coil_b_nega_ter = M2D.assign_coil(assignment=coil_b_nega, conductors_number=1, polarity='Negative', name="coil_b_nega_ter")
coil_c_posi_ter = M2D.assign_coil(assignment=coil_c_posi, conductors_number=1, polarity='Positive', name="coil_c_posi_ter")
coil_c_nega_ter = M2D.assign_coil(assignment=coil_c_nega, conductors_number=1, polarity='Negative', name="coil_c_nega_ter")



# 2. create windings and add the coil terminals to them
# - winding current can be changed later just by updating the current_a/b/c Maxwell variables
winding_a = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current="current_a", resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_a")    # create winding
winding_b = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current="current_b", resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_b")    # create winding
winding_c = M2D.assign_winding(assignment=None, winding_type='Current', is_solid=False, current="current_c", resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_c")    # create winding


M2D.add_winding_coils(assignment=winding_a.name, coils=[coil_a_posi_ter.name, coil_a_nega_ter.name])   # add coil to winding
M2D.add_winding_coils(assignment=winding_b.name, coils=[coil_b_posi_ter.name, coil_b_nega_ter.name])   # add coil to winding
M2D.add_winding_coils(assignment=winding_c.name, coils=[coil_c_posi_ter.name, coil_c_nega_ter.name])   # add coil to winding

In [ ]:
## Force and torque parameters ##

# rotor torque parameter
rotor_torque = M2D.assign_torque(assignment=[magnet_1, magnet_2, magnet_3, magnet_4, rotor], coordinate_system='Global', is_positive=True, is_virtual=True, axis='Z', torque_name="rotor_torque")


# rotor x/y force parameter
rotor_force = M2D.assign_force(assignment=[magnet_1, magnet_2, magnet_3, magnet_4, rotor], coordinate_system='Global', is_virtual=True, force_name="rotor_force")

In [ ]:
## Angle sweep: analyze, export, rotate ##

# sweep 0 to 360 degrees in 10-degree steps, re-analyzing at each step
sim_step = 10
for angle_deg in range(0, 360, sim_step):

    # electrical angle at this rotor position (offset by 90 deg)
    theta_deg = 90 + angle_deg
    theta_rad = math.radians(theta_deg)

    # current density for each coil (120-degree phase shift)

    J_a = J_rated * math.cos(theta_rad)                    # coil 1 (U phase)  // a
    J_b = J_rated * math.cos(theta_rad - 2*math.pi/3)      # coil 2 (V phase)  // b
    J_c = J_rated * math.cos(theta_rad + 2*math.pi/3)      # coil 3 (W phase)  // c

    # terminal area (mm^2)
    terminal_area_a = coil_a_posi.faces[0].area # a
    terminal_area_b = coil_b_posi.faces[0].area # b
    terminal_area_c = coil_c_posi.faces[0].area # c

    # actual current per coil (current = current density x area)
    total_current_a = J_a * terminal_area_a # a
    total_current_b = J_b * terminal_area_b # b
    total_current_c = J_c * terminal_area_c # c

    # write the computed current values into the Maxwell design variables
    M2D["current_a"] = f"{total_current_a}A" # a
    M2D["current_b"] = f"{total_current_b}A" # b
    M2D["current_c"] = f"{total_current_c}A" # c

    # print each coil's numbers to check them as the sweep runs
    print(f"[coil a] area: {terminal_area_a:.2f} mm^2, current density: {J_a:.3f} A/mm^2, current: {total_current_a:.3f} A, phase: 0deg")   # a
    print(f"[coil b] area: {terminal_area_b:.2f} mm^2, current density: {J_b:.3f} A/mm^2, current: {total_current_b:.3f} A, phase: 120deg") # b
    print(f"[coil c] area: {terminal_area_c:.2f} mm^2, current density: {J_c:.3f} A/mm^2, current: {total_current_c:.3f} A, phase: -120deg") # c

    # run the analysis
    my_setup.analyze()

    # - script-recorded report creation
    # - data table, with units set to NewtonMeter / Newton / Newton
    table_name = "data_table"
    
    oModule = oDesign.GetModule("ReportSetup")
    oModule.CreateReport(f"{table_name}", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [], 
        [
            "depth:="		, ["All"],
            "slot_number:="		, ["Nominal"],
            "r_so:="		, ["Nominal"],
            "r_si:="		, ["Nominal"],
            "th_core:="		, ["Nominal"],
            "w_teeth:="		, ["Nominal"],
            "slot_ratio:="		, ["Nominal"],
            "shoe_1:="		, ["Nominal"],
            "shoe_2:="		, ["Nominal"],
            "c_w:="			, ["Nominal"],
            "c_h:="			, ["Nominal"],
            "l_c:="			, ["Nominal"],
            "r_rotor_outer:="	, ["Nominal"],
            "num_pole:="		, ["Nominal"],
            "m_w:="			, ["Nominal"],
            "m_th:="		, ["Nominal"],
            "l_m:="			, ["Nominal"],
            "gamma:="		, ["Nominal"],
            "N:="			, ["Nominal"],
            "J_rated:="		, ["Nominal"],
            "current_a:="		, ["Nominal"],
            "current_b:="		, ["Nominal"],
            "current_c:="		, ["Nominal"]
        ], 
        [
            "X Component:="		, "depth",
            "Y Component:="		, ["rotor_torque.Torque", "rotor_force.Force_x", "rotor_force.Force_y"]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:rotor_torque.Torque:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "NewtonMeter"
                    ]
                ]
            ]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:rotor_force.Force_x:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "Newton"
                    ]
                ]
            ]
        ])
    oModule.ChangeProperty(
        [
            "NAME:AllTabs",
            [
                "NAME:Data Filter",
                [
                    "NAME:PropServers", 
                    f"{table_name}:rotor_force.Force_y:Curve1"
                ],
                [
                    "NAME:ChangedProps",
                    [
                        "NAME:Units",
                        "Value:="		, "Newton"
                    ]
                ]
            ]
        ])
    
    csv_name = f"data_torque_{angle_deg}_deg_e.csv"
    csv_path = os.path.join(dir, csv_name)

    oModule.ExportToFile(f"{table_name}", csv_path, False)

    oModule.DeleteReports([f"{table_name}"])

    # rotate the stator by the step angle (absolute coordinates)
    # - the rotor magnets are magnetized in global-coordinate directions, so the
    #   stator is rotated instead of the rotor
    # - rotating the stator backward (CW) is equivalent to the rotor turning
    #   forward (CCW)
    M2D.modeler.rotate(assignment=[stator, coil_a_posi, coil_a_nega, coil_b_posi, coil_b_nega, coil_c_posi, coil_c_nega], axis='Z', angle=f'-({sim_step}/(num_pole/2))deg')


In [ ]:
## Combine results, convert to global frame, and plot ##

# read the per-angle data back in from the CSVs written during the sweep
angles = list(range(0, 360, sim_step))

torque_values = []
force_x_values = []
force_y_values = []

for angle_deg in angles:
    csv_name = f"data_torque_{angle_deg}_deg_e.csv"
    csv_path = os.path.join(dir, csv_name)
    df = pd.read_csv(csv_path)
    # read the last row (column order: depth, Torque, Force_x, Force_y)
    torque_values.append(float(df.iloc[-1, 1]))
    force_x_values.append(float(df.iloc[-1, 2]))
    force_y_values.append(float(df.iloc[-1, 3]))

# build the combined CSV file
df_combined = pd.DataFrame({
    "Rotor_Angle_deg": angles,
    "Torque_Nm": torque_values,
    "Force_X_N": force_x_values,
    "Force_Y_N": force_y_values,
})
# saved flat, next to the notebook -- same convention as Images/ (the per-angle
# scratch CSVs above stay in dir since they get deleted right after combining)
combined_csv_path = "Week6_ex3_torque_force_combined.csv"
df_combined.to_csv(combined_csv_path, index=False)
print(f"combined CSV saved: {combined_csv_path}")

# clean up the per-angle CSV files now that they're combined
for angle_deg in angles:
    csv_name = f"data_torque_{angle_deg}_deg_e.csv"
    csv_path = os.path.join(dir, csv_name)
    if os.path.exists(csv_path):
        os.remove(csv_path)
print(f"removed {len(angles)} per-angle CSV files")

# apply the rotation matrix: rotor frame -> global frame
# the Force X/Y measured in the simulation is in the rotating rotor frame
# convert to the global frame: [Fx_g, Fy_g] = R(theta) @ [Fx_r, Fy_r]
force_x_global = []
force_y_global = []
for i, angle_deg in enumerate(angles):
    theta = np.radians(angle_deg/(num_pole/2))
    fx_r = force_x_values[i]
    fy_r = force_y_values[i]
    fx_g = fx_r * np.cos(theta) - fy_r * np.sin(theta)
    fy_g = fx_r * np.sin(theta) + fy_r * np.cos(theta)
    force_x_global.append(fx_g)
    force_y_global.append(fy_g)

# update the Force columns in df_combined with the global-frame values, then re-save
df_combined["Force_X_N"] = force_x_global
df_combined["Force_Y_N"] = force_y_global
df_combined.to_csv(combined_csv_path, index=False)
print("rotation matrix applied (rotor frame -> global frame), CSV re-saved")

# read the combined CSV back in for plotting
df_plot = pd.read_csv(combined_csv_path)
angles_plot = df_plot["Rotor_Angle_deg"].tolist()
torque_plot  = df_plot["Torque_Nm"].tolist()
force_x_plot = df_plot["Force_X_N"].tolist()
force_y_plot = df_plot["Force_Y_N"].tolist()

print("angle [deg]:", angles_plot)
print("Torque [N*m]:", [f"{v:.4f}" for v in torque_plot])
print("Force X [N]:",  [f"{v:.4f}" for v in force_x_plot])
print("Force Y [N]:",  [f"{v:.4f}" for v in force_y_plot])

# plot
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
fig.suptitle("4P3S IPM Motor - Simulation Results", fontsize=14, fontweight="bold")

axes[0].plot(angles_plot, torque_plot, "b-o", linewidth=2, markersize=6, label="Torque")
axes[0].set_ylabel("Torque [N*m]")
axes[0].set_title("Rotor Torque vs Rotor Angle")
axes[0].grid(True, linestyle="--", alpha=0.7)
axes[0].legend()

axes[1].plot(angles_plot, force_x_plot, "r-s", linewidth=2, markersize=6, label="Force X")
axes[1].set_ylabel("Force X [N]")
axes[1].set_title("Rotor Force X vs Rotor Angle")
axes[1].grid(True, linestyle="--", alpha=0.7)
axes[1].legend()

axes[2].plot(angles_plot, force_y_plot, "g-^", linewidth=2, markersize=6, label="Force Y")
axes[2].set_xlabel("Rotor Angle [deg]")
axes[2].set_ylabel("Force Y [N]")
axes[2].set_title("Rotor Force Y vs Rotor Angle")
axes[2].grid(True, linestyle="--", alpha=0.7)
axes[2].legend()

axes[2].set_xticks(angles_plot)

plt.tight_layout()

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week6_ex3_torque_force_vs_angle.png")
plt.show()

In [ ]:
## Save project ##

M2D.save_project()